# Graph Feature Extraction — No Teacher Soft Labels

This notebook converts molecular SMILES into graph dictionaries without teacher predictions. **The current implementation still requires `SMILES` and `y_true` input columns** and saves an empty `(1, 0)` `y_soft` tensor for every graph. It is therefore not a target-free inference pipeline.

**Input:** `data-set/input/label_free.csv` (set the relative path below to your actual input file). **Outputs:** `graph_data_with_soft_labels.pt`, `feature_scalers.pkl`, and `metadata.txt` under `data-set/features/label_free/`. The output names and graph dictionary keys are retained for compatibility.

Run this notebook on the intended training partition when fitting feature scalers. The notebook fits new atom/bond scalers on its entire input and does **not** implement a separate transform-only mode for held-out data. Padding, graph connectivity, and feature definitions follow the source implementation.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import pickle

from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

# --- Feature encoding (original feature definitions and dimensions) ---
def one_of_k_encoding_unk(x, allowable_set):
    """One-hot encoding; map unsupported values to the final category."""
    if x not in allowable_set:
        x = allowable_set[-1]
    return [x == s for s in allowable_set]

def extract_atom_features(atom):
    """Return the original atom feature vector without changing its feature order."""
    atom_idx = int(atom.GetIdx())
    allowable_elements = ['C','N','O','S','F','H','Si','P','Cl','Br',
                          'Li','Na','K','Mg','Ca','Fe','As','Al','I','B',
                          'V','Tl','Sb','Sn','Ag','Pd','Co','Se','Ti','Zn',
                          'Ge','Cu','Au','Ni','Cd','Mn','Cr','Pt','Hg','Pb']
    
    basic_features = one_of_k_encoding_unk(atom.GetSymbol(), allowable_elements) + \
                     one_of_k_encoding_unk(atom.GetDegree(), [0,1,2,3,4,5]) + \
                     one_of_k_encoding_unk(atom.GetTotalNumHs(), [0,1,2,3,4]) + \
                     one_of_k_encoding_unk(atom.GetImplicitValence(), [0,1,2,3,4,5]) + \
                     [atom.GetIsAromatic()]
    
    extra_features = [
        atom.GetAtomicNum() / 100.0,
        atom.GetMass() / 100.0,
        atom.GetFormalCharge(),
        atom.GetNumRadicalElectrons(),
        atom.GetNumExplicitHs(),
    ]
    
    if atom.HasProp('_GasteigerCharge'):
        extra_features.append(float(atom.GetProp('_GasteigerCharge')))
    else:
        extra_features.append(0.0)
    
    extra_features += one_of_k_encoding_unk(atom.GetHybridization(), list(Chem.rdchem.HybridizationType.values))
    extra_features += one_of_k_encoding_unk(atom.GetChiralTag(), list(Chem.rdchem.ChiralType.values))
    
    ring_features = [atom.IsInRing()]
    for ring_size in range(3, 8):
        ring_features.append(atom.IsInRingSize(ring_size))
    extra_features += ring_features
    
    return np.array(basic_features + extra_features, dtype=float)

def extract_bond_features(bond):
    """Encode bond type, ring membership, conjugation, and stereochemistry."""
    bond_type = bond.GetBondType()
    bond_features = [
        bond_type == Chem.rdchem.BondType.SINGLE,
        bond_type == Chem.rdchem.BondType.DOUBLE,
        bond_type == Chem.rdchem.BondType.TRIPLE,
        bond_type == Chem.rdchem.BondType.AROMATIC,
        bond.IsInRing(),
        bond.GetIsConjugated(),
        bond.GetStereo() != Chem.rdchem.BondStereo.STEREONONE
    ]
    return np.array(bond_features, dtype=float)

def extract_molecule_global_features(mol):
    """Combine eight molecular descriptors with a 64-bit Morgan fingerprint."""
    physicochem_features = [
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Descriptors.TPSA(mol),
        Descriptors.NumHAcceptors(mol),
        Descriptors.NumHDonors(mol),
        rdMolDescriptors.CalcNumRotatableBonds(mol),
        rdMolDescriptors.CalcNumHeteroatoms(mol),
        rdMolDescriptors.CalcFractionCSP3(mol),
    ]
    
    morgan_fingerprint = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=64)
    fp_features = list(morgan_fingerprint)
    
    return physicochem_features + fp_features

class MolGraphBuilder:
    """Build padded atom graphs with the original node and edge conventions."""
    def __init__(self, atom_scaler=None, bond_scaler=None):
        self.atom_scaler = atom_scaler
        self.bond_scaler = bond_scaler
    
    def build_graph(self, smiles, max_atoms, use_edge_features=True, use_global_features=True):
        """Return a graph dictionary, or None when SMILES cannot be parsed."""
        parsed_mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None
        mol = Chem.AddHs(parsed_mol) if parsed_mol is not None else None
        if mol is None: 
            return None
        
        atom_features_list = [extract_atom_features(atom) for atom in mol.GetAtoms()]
        num_atoms = len(atom_features_list)
        atom_feature_dim = len(atom_features_list[0]) if num_atoms > 0 else 0
        
        padded_atom_features = np.zeros((max_atoms, atom_feature_dim), dtype=float)
        if num_atoms > 0:
            padded_atom_features[:num_atoms] = np.vstack(atom_features_list)
            if self.atom_scaler is not None:
                padded_atom_features = self.atom_scaler.transform(padded_atom_features)
        
        edge_index = []
        edge_attr = []
        
        for bond in mol.GetBonds():
            atom_i = int(bond.GetBeginAtomIdx())
            atom_j = int(bond.GetEndAtomIdx())
            edge_index.append([atom_i, atom_j])
            edge_index.append([atom_j, atom_i])
            
            if use_edge_features:
                bond_feat = extract_bond_features(bond)
                edge_attr.append(bond_feat)
                edge_attr.append(bond_feat)
        
        for atom_idx in range(num_atoms):
            edge_index.append([atom_idx, atom_idx])
            if use_edge_features:
                bond_feat_dim = len(edge_attr[0]) if edge_attr else 7
                self_loop_bond_feat = np.zeros(bond_feat_dim)
                edge_attr.append(self_loop_bond_feat)
        
        edge_index_tensor = torch.tensor(edge_index).t().contiguous()
        edge_attr_tensor = None
        
        if use_edge_features and len(edge_attr) > 0:
            edge_attr_tensor = torch.tensor(np.asarray(edge_attr), dtype=torch.float)
            if self.bond_scaler is not None:
                edge_attr_tensor = torch.tensor(self.bond_scaler.transform(edge_attr_tensor.numpy()), dtype=torch.float)
        
        global_features_tensor = None
        if use_global_features:
            try:
                global_features = extract_molecule_global_features(mol)
                global_features_tensor = torch.tensor(global_features, dtype=torch.float).view(1, -1)
            except Exception as e:
                global_features_tensor = None
        
        return {
            'x': torch.tensor(padded_atom_features, dtype=torch.float),
            'edge_index': edge_index_tensor,
            'edge_attr': edge_attr_tensor,
            'u': global_features_tensor,
            'smiles': smiles,
            'num_atoms': num_atoms
        }


# --- Graph construction and artifact export ---
def extract_mol_features_with_soft_labels(
    df,
    soft_label_cols,
    output_dir="mol_graph_features_with_soft_labels",
    max_atoms=None,
    use_edge_features=True,
    use_global_features=True
):
    """Fit graph feature scalers and export graphs using the original schema."""
    required_columns = ["SMILES", "y_true", *soft_label_cols]
    missing = [column for column in required_columns if column not in df.columns]
    if missing:
        raise ValueError(f"Missing required input columns: {missing}")
    if df.empty:
        raise ValueError("The input CSV contains no rows.")
    os.makedirs(output_dir, exist_ok=True)
    
    if max_atoms is None:
        atom_counts = []
        for smiles in df.SMILES:
            parsed_mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None
            mol = Chem.AddHs(parsed_mol) if parsed_mol is not None else None
            atom_count = mol.GetNumAtoms() if mol is not None else 0
            atom_counts.append(atom_count)
        max_atoms = max(atom_counts) + 5
        print(f"Auto-calculated max atoms: {max_atoms}")
    
    print("Extracting atom features for normalization...")
    all_atom_features = []
    for smiles in tqdm(df["SMILES"], desc="Processing Atoms"):
        parsed_mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None
        mol = Chem.AddHs(parsed_mol) if parsed_mol is not None else None
        if mol is not None:
            for atom in mol.GetAtoms():
                all_atom_features.append(extract_atom_features(atom))
    
    # These scalers are fitted on all rows in this input: supply training data only.
    if not all_atom_features:
        raise ValueError("No valid SMILES were available for atom feature scaling.")
    atom_scaler = StandardScaler()
    atom_scaler.fit(np.array(all_atom_features))
    
    bond_scaler = None
    if use_edge_features:
        all_bond_features = []
        print("Extracting bond features for normalization...")
        for smiles in tqdm(df["SMILES"], desc="Processing Bonds"):
            parsed_mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None
            mol = Chem.AddHs(parsed_mol) if parsed_mol is not None else None
            if mol is not None and mol.GetNumBonds() > 0:
                for bond in mol.GetBonds():
                    all_bond_features.append(extract_bond_features(bond))
        
        if len(all_bond_features) > 0:
            bond_scaler = StandardScaler()
            bond_scaler.fit(np.array(all_bond_features))
    
    graph_builder = MolGraphBuilder(atom_scaler, bond_scaler)
    graph_data_list = []
    error_count = 0
    max_error_display = 20
    
    print("Building molecule graphs (with soft labels)...")
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Building Graphs"):
        try:
            graph_data = graph_builder.build_graph(
                row["SMILES"],
                max_atoms,
                use_edge_features,
                use_global_features
            )
            if graph_data is None:
                continue
            
            if 'y_true' in row:
                graph_data['y'] = torch.tensor([row['y_true']], dtype=torch.float)
            else:
                raise ValueError("DataFrame missing 'y_true' target column")
            
            soft_labels = []
            for col in soft_label_cols:
                if col in row:
                    soft_labels.append(row[col])
                else:
                    raise ValueError(f"Missing soft label column: '{col}'")
            # An empty soft_label_cols list produces the existing (1, 0) tensor.
            graph_data['y_soft'] = torch.tensor(soft_labels, dtype=torch.float).view(1, -1)
            
            graph_data_list.append(graph_data)
        
        except Exception as e:
            error_count += 1
            if error_count <= max_error_display:
                print(f"Error processing SMILES {row['SMILES']}: {str(e)}")
            elif error_count == max_error_display + 1:
                print(f"Max error display ({max_error_display}) reached, hiding subsequent errors...")
    
    print(f"Graph building completed | Errors: {error_count} | Valid samples: {len(graph_data_list)}")
    
    print(f"Saving data to {output_dir}...")
    torch.save(graph_data_list, os.path.join(output_dir, "graph_data_with_soft_labels.pt"))
    with open(os.path.join(output_dir, "feature_scalers.pkl"), 'wb') as f:
        pickle.dump({
            'atom_scaler': atom_scaler,
            'bond_scaler': bond_scaler,
            'max_atoms': max_atoms
        }, f)
    with open(os.path.join(output_dir, "metadata.txt"), 'w') as f:
        f.write(f"max_atoms={max_atoms}\n")
        f.write(f"use_edge_features={use_edge_features}\n")
        f.write(f"use_global_features={use_global_features}\n")
        f.write(f"num_valid_samples={len(graph_data_list)}\n")
        f.write(f"soft_label_columns={', '.join(soft_label_cols) if soft_label_cols else 'None'}\n")
    
    print("Molecule graph feature extraction completed!")
    return output_dir

if __name__ == "__main__":
    # Paths are relative to the repository root / current working directory.
    input_csv_path = os.path.join("data-set", "input", "label_free.csv")
    output_feature_dir = os.path.join("data-set", "features", "label_free")

    df = pd.read_csv(input_csv_path)

    # No teacher prediction columns are included in this variant.
    soft_label_columns = []

    feature_directory = extract_mol_features_with_soft_labels(
        df=df,
        soft_label_cols=soft_label_columns,
        output_dir=output_feature_dir,
        use_edge_features=True,
        use_global_features=True
    )
    print(f"All features saved to: {feature_directory}")
